# Deploy an XGBoost model on TrueFoundry with MLServer (Seldon), no inference code

- Flow: **train → log model → package → build & push image → deploy → infer**
- Inference code: none — MLServer's `XGBoostModel` runtime
- Image: built by `mlserver build`, pushed to registry first, never built by TrueFoundry
- Docs: https://docs.seldon.ai/mlserver, https://mlserver.readthedocs.io/en/latest/runtimes/xgboost.html

---

## 0. Setup

> **Kernel: `Python 3.12 (TFY MLServer)`** — venv at `~/work/.venvs/tfy-mlserver-py312`.
> Also not a preference: `mlserver` is `requires_python <3.13`, so `mlserver build` cannot run
> on 3.14. Kept separate from the Triton env, which pins numpy < 2.

In [1]:
# %pip install -U truefoundry scikit-learn xgboost mlserver mlserver-xgboost requests

In [2]:
import json
import os
import sys
import time
from pathlib import Path
from pprint import pprint

import requests
from sklearn.datasets import load_iris
from xgboost import XGBClassifier

from truefoundry.ml import get_client
from truefoundry.common.session import Session
from truefoundry_sdk import TrueFoundry

# `!` inherits the notebook process PATH, which lacks the kernel venv's bin/.
os.environ["PATH"] = f"{Path(sys.executable).parent}{os.pathsep}{os.environ['PATH']}"

In [3]:
# devtest: XGBoost deployment needs mlfoundry-server#1196, not on live-demo yet.
# Run `tfy login --host https://internal.devtest.truefoundry.tech --relogin` first.
TFY_HOST = "https://internal.devtest.truefoundry.tech"
ML_REPO = "default"
CLUSTER = "tfy-usea1-devtest"
WORKSPACE = "seldon-tf"
DOMAIN = "tfy-usea1-ctl.devtest.truefoundry.tech"  # per-cluster: tfy-usea1-devtest
WS_FQN = f"{CLUSTER}:{WORKSPACE}"

DOCKER_USER = "mkozubal"
TAG = "1"
PORT = 8080
READY_TIMEOUT = 300

NAME = "iris-xgb-mlserver"
SVC = f"{NAME}-svc"
IMAGE = f"{DOCKER_USER}/{SVC}:{TAG}"
HOST = f"{SVC}-{WORKSPACE}-{PORT}.{DOMAIN}"

## 1. Train and save

In [4]:
Path(SVC).mkdir(exist_ok=True)

X, y = load_iris(return_X_y=True)
model = XGBClassifier(n_estimators=10).fit(X, y)
# Not joblib: XGBoostModel loads through `xgboost.Booster`, which wants the native
# format — `model.json`, `model.ubj` or `model.bst`.
model.save_model(Path(SVC, "model.json"))

## 2. Describe the model for MLServer

No schema and no generated server. `model-settings.json` names the runtime, and MLServer's
`XGBoostModel` loads the booster itself. The class is `XGBoostModel` — not `XGBoostRuntime`.

In [5]:
Path(SVC, "model-settings.json").write_text(json.dumps({
    "name": SVC,
    "implementation": "mlserver_xgboost.XGBoostModel",
    "parameters": {"uri": "./model.json"},
}, indent=2))

Path(SVC, "requirements.txt").write_text("mlserver-xgboost==1.7.1\n")

24

## 3. Log to the registry 
- no schema
- versioning and lineage only: no framework, so no schema and no `deploy-init`.
`mlserver build` is what turns the folder into a server.

In [6]:
client = get_client()

FQN = client.log_model(ml_repo=ML_REPO, name=NAME, model_file_or_folder=SVC).fqn
print(FQN)

[truefoundry.ml] 2026-09-29T16:01:27+0100 INFO Logged in to 'https://internal.devtest.truefoundry.tech' as 'maciej.kozubal@truefoundry.com'
[truefoundry.ml.log_types.artifacts.model] 2026-09-29T16:01:27+0100 INFO Logging model and additional files, this might take a while ...
[truefoundry.ml.log_types.artifacts.model] 2026-09-29T16:01:28+0100 INFO Adding model file/folder to model version content
[truefoundry.ml.log_types.artifacts.utils] 2026-09-29T16:01:28+0100 INFO Adding contents of iris-xgb-mlserver-svc/ to /./
[truefoundry.ml.log_types.artifacts.model] 2026-09-29T16:01:28+0100 INFO Packaging and uploading files to remote with Total Size: 0.025626 MB
[truefoundry.ml] 2026-09-29T16:01:28+0100 WARNING Detected Jupyter Environment. Install `ipywidgets` to display live progress bars.
[truefoundry.ml] 2026-09-29T16:01:28+0100 INFO Uploading /Users/maciej/work/seldon-tf/tf-workloads/core2-iris-no-inference-code/notebooks/iris-xgb-mlserver-svc/model.json to model.json
[truefoundry.ml] 20

## 4. Build and push the image

`mlserver build` writes its own Dockerfile (base `seldonio/mlserver:*-slim`) and its `COPY . .`
**bakes the model into the image** — the Seldon way, so no `artifacts_download` when we deploy.

It has no `--platform` flag, but it shells out to `docker build`, so `DOCKER_DEFAULT_PLATFORM`
gets us the amd64 image the cluster needs.

In [7]:
os.environ["DOCKER_DEFAULT_PLATFORM"] = "linux/amd64"

!cd {SVC} && mlserver build . -t {IMAGE}
!docker push {IMAGE}

2026-09-29 16:01:31,882 [mlserver] INFO - Building Docker image with tag mkozubal/iris-xgb-mlserver-svc:1
2026-09-29 16:01:31,883 [mlserver] INFO - Writing Dockerfile in /var/folders/4j/mhjp_0j100j5g2zkj93g0kc80000gn/T/tmpldc_zfpg/Dockerfile
2026-09-29 16:01:31,883 [mlserver] INFO - Writing .dockerignore in /var/folders/4j/mhjp_0j100j5g2zkj93g0kc80000gn/T/tmpldc_zfpg/Dockerfile.dockerignore
...
d677d442: Waiting 
10e6c2a6: Waiting 
50da25b1: Waiting 
d5ce49f6: Pushed   543.1MB/543.1MBis-sk-mlserver-svc Pushing  1.788kB/1.788kB1: digest: sha256:ea4593f11d4867cbfdbda6fa5a742ccf29ece560d2fd9d62a6432eb497c5a872 size: 856


## 5. Deploy as a service

- image: prebuilt (`"image": {"type": "image", ...}`)
- model: already baked into the image, so no `artifacts_download`
- auth reuses the `tfy login` session (no PAT in the notebook)
- `create_or_update`: returns a deployment, so delete later with `app.data.application_id`

In [8]:
tfy = TrueFoundry(base_url=TFY_HOST, api_key=Session.new().access_token)

app = tfy.applications.create_or_update(manifest={
    "name": SVC, "type": "service", "workspace_fqn": WS_FQN,
    "image": {"type": "image", "image_uri": IMAGE},
    "ports": [{"port": PORT, "protocol": "TCP", "expose": True,
               "app_protocol": "http", "host": HOST}],
    "resources": {"cpu_request": 0.5, "cpu_limit": 0.5,
                  "memory_request": 1000, "memory_limit": 1000,
                  "ephemeral_storage_request": 2000, "ephemeral_storage_limit": 2000},
    "replicas": 1,
})
print(SVC, app.data.application_id)

iris-xgb-mlserver-svc rlzitnm900341wue7ajrjvt5


## 6. Call the model

- mlserver v2 on port 8080: `POST /v2/models/<name>/infer`
- `data` is flat row-major alongside `shape` (Triton also accepts nested)
- input name is the runtime's method: `predict`
- output is one row of **per-class probabilities** (`multi:softprob`), not a label — unlike the sklearn runtime

In [10]:
deadline = time.time() + READY_TIMEOUT
while time.time() < deadline:
    try:
        if requests.get(f"https://{HOST}/v2/models/{SVC}/ready", timeout=5).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(5)
else:
    raise TimeoutError(f"{SVC} not ready after {READY_TIMEOUT}s")

print(SVC, "ready")

iris-xgb-mlserver-svc ready


In [11]:
r = requests.post(
    f"https://{HOST}/v2/models/{SVC}/infer",
    json={"inputs": [{"name": "predict", "shape": [1, 4], "datatype": "FP32",
                      "data": [5.1, 3.5, 1.4, 0.2]}]},
    timeout=30,
)
print(f"{r.status_code = }\n")
pprint(json.loads(r.text))

r.status_code = 200

{'id': 'e93e4d8a-0773-42a7-b2fa-e8ad26847a4f',
 'model_name': 'iris-xgb-mlserver-svc',
 'outputs': [{'data': [0],
              'datatype': 'INT64',
              'name': 'predict',
              'parameters': {'content_type': 'np'},
              'shape': [1, 1]}],
 'parameters': {}}


The same call as a copy-pasteable `curl`, for testing outside the notebook:

In [12]:
print(f'''curl -X POST https://{HOST}/v2/models/{SVC}/infer \\
  -H 'Content-Type: application/json' \\
  -d '{{"inputs": [{{"name": "predict", "shape": [1, 4], "datatype": "FP32", "data": [5.1, 3.5, 1.4, 0.2]}}]}}' ''')

curl -X POST https://iris-xgb-mlserver-svc-seldon-tf-8080.tfy-usea1-ctl.devtest.truefoundry.tech/v2/models/iris-xgb-mlserver-svc/infer \
  -H 'Content-Type: application/json' \
  -d '{"inputs": [{"name": "predict", "shape": [1, 4], "datatype": "FP32", "data": [5.1, 3.5, 1.4, 0.2]}]}' 


## Cleanup
- service, local package, model file
- model version stays — a re-run logs a new one
- image left alone — `docker build`/`push` overwrite tag `1` in place

In [ ]:
# service
APP_ID = app.data.application_id
try:
    tfy.applications.delete(id=APP_ID)
except Exception:
    tfy.applications.cancel_deployment(
        id=APP_ID, deployment_id=tfy.applications.get(id=APP_ID).data.last_deployment_id)
    tfy.applications.delete(id=APP_ID)
print("deleted", SVC)

# local package
!rm -rf {SVC}
